# core

> Kernel processes, channels, routing, and the gateway server

In [ ]:
#| default_exp core

In [ ]:
#| export
import asyncio, json, logging, os, signal, subprocess, sys, time, uuid
from collections import deque
from contextlib import asynccontextmanager
from pathlib import Path
from tempfile import mkdtemp
import zmq, zmq.asyncio
from jupywire.connect import write_connection_file
from fastcore.basics import xdumps, revive_dates
from jupywire.session import Session, pack_frames, unpack_frames
from starlette.applications import Starlette
from starlette.responses import JSONResponse, Response
from starlette.routing import Route, WebSocketRoute
from starlette.websockets import WebSocketDisconnect

In [ ]:
import httpx, struct, time
from fastcore.test import test_eq, expect_fail
from websockets.sync.client import connect as ws_connect
from websockets.exceptions import InvalidStatus

In [ ]:
#| export
log = logging.getLogger('jupygate')

## Wire formats

Kernels communicate with the gateway over zmq using the [Jupyter messaging protocol](https://jupyter-client.readthedocs.io/en/latest/messaging.html). Messages contain `header`, `parent_header`, `metadata`, `content`, and optional binary `buffers`.

Websocket clients use Jupyter's legacy websocket protocol. It carries the same message fields and adds `channel` to select `shell`, `iopub`, `stdin`, or `control`. Clients that implement this protocol can connect without a jupygate-specific codec.

The functions below convert between message dictionaries and websocket frames. They work on dictionaries and bytes without a kernel, sockets, or an event loop.

### Text frames


Messages without buffers use JSON text frames. `xdumps` serializes header values such as datetimes and UUIDs. On decoding, `revive_dates` restores datetime values in `header` and `parent_header`.

Content strings remain strings even when they look like dates. This matches jupyter_client's handling of message headers over zmq.


In [ ]:
#| export
def dumps(msg:dict)->str:
    "Encode a Jupyter message dict (no buffers) as a websocket text frame."
    return xdumps(msg)

def loads(s:str|bytes)->dict:
    "Decode a websocket text frame back into a message dict, reviving header dates."
    msg = json.loads(s)
    for k in ('header','parent_header'):
        if isinstance(msg.get(k), dict): msg[k] = revive_dates(msg[k])
    return msg


We'll use an `execute_request` throughout the examples. Jupywire's `Session` builds the same headers as jupyter_client:

In [ ]:
session = Session(key=b"secret")
req = session.msg('execute_request', dict(code='6*7', silent=False))
req['channel'] = 'shell'
sorted(req)

['channel',
 'content',
 'header',
 'metadata',
 'msg_id',
 'msg_type',
 'parent_header']

In [ ]:
frame = dumps(req)
frame[:120]

'{"header": {"msg_id": "66293220-2297-492d-8ec6-a81726f4f04b_37680_0", "msg_type": "execute_request", "username": "jhowar'

The round trip preserves everything, including the parsed datetime in the header.

In [ ]:
back = loads(frame)
test_eq(back['header'], req['header'])
test_eq(back['content'], req['content'])
back['header']['date']

datetime.datetime(2026, 9, 9, 8, 43, 37, 206266, tzinfo=datetime.timezone.utc)

### Binary frames


Messages with binary buffers use a binary websocket frame. A comm message containing array data is one example.

The frame begins with a four-byte, big-endian count named `nbufs`. This counts the JSON part and all binary buffers. Next come `nbufs` four-byte offsets, measured from the start of the frame. The remaining bytes contain the JSON part followed by each buffer.

This is jupyter_server's legacy binary layout. The wrappers use `pack_frames` and `unpack_frames` from `jupywire.session` for framing. They encode the message fields separately from its `buffers`.

In [ ]:
#| export
def dumps_binary(msg:dict)->bytes:
    "Encode a message with `buffers` as a legacy-protocol binary frame."
    msg = dict(msg)
    buffers = msg.pop('buffers')
    return pack_frames(dumps(msg).encode('utf-8'), buffers)

def loads_binary(bmsg:bytes)->dict:
    "Decode a legacy-protocol binary frame back into a message dict with `buffers`."
    body, buffers = unpack_frames(bmsg)
    msg = loads(body)
    msg['buffers'] = buffers
    return msg


This comm message has two buffers. The first 16 bytes contain a count of three parts and all three offsets. The JSON message is the first part:

In [ ]:
comm = session.msg('comm_msg', dict(comm_id='abc', data={}))
comm['channel'] = 'iopub'
comm['buffers'] = [b'\x00'*8, b'payload']
bframe = dumps_binary(comm)
struct.unpack('!4I', bframe[:16])

(3, 16, 428, 436)

In [ ]:
back = loads_binary(bframe)
test_eq(back['buffers'], [b'\x00'*8, b'payload'])
test_eq(back['content'], comm['content'])
back['channel']

'iopub'

### One entry point each way


`to_frame` selects binary encoding when the message has nonempty `buffers`. Otherwise it returns text. `from_frame` selects the decoder by input type: `str` for text, `bytes` or `bytearray` for binary. Its result always includes a `buffers` list.

In [ ]:
#| export
def to_frame(msg:dict)->str|bytes:
    "Encode `msg` for the websocket: JSON text frame, or binary frame if it carries buffers."
    return dumps_binary(msg) if msg.get('buffers') else dumps({k:v for k,v in msg.items() if k!='buffers'})

def from_frame(data:str|bytes)->dict:
    "Decode a websocket frame (text or binary) into a message dict; `buffers` is always present."
    msg = loads_binary(data) if isinstance(data, (bytes,bytearray)) else loads(data)
    msg.setdefault('buffers', [])
    return msg

In [ ]:
test_eq(from_frame(to_frame(req))['content'], req['content'])
test_eq(from_frame(to_frame(comm))['buffers'], comm['buffers'])
type(to_frame(req)), type(to_frame(comm))

(str, bytes)

An empty `buffers` list uses a text frame. Clients can include the key in every message without forcing binary encoding:

In [ ]:
plain = dict(req, buffers=[])
assert isinstance(to_frame(plain), str)
from_frame(to_frame(plain))['buffers']

[]

## Kernel processes

Clients specify a kernel command, environment, and working directory. The gateway writes a connection file and starts the process. It also terminates the process when the client deletes the kernel.

There is no kernelspec lookup or environment whitelist. The gateway runs the supplied command. This supports launch requirements such as solveit's custom arguments and environment, including its gosu/sudo wrapper for another user.

This API permits arbitrary code execution, like a Jupyter kernel. It requires trusted clients. Configure authentication before exposing the gateway beyond a trusted local environment.

### Connection files


A Jupyter connection file contains five ports, an IP address, a transport, and an HMAC signing key. `make_connection` uses jupywire's `write_connection_file` to allocate free ports and write the JSON file. It returns the file path and connection information. `KernelChannels` uses that information to connect.

In [ ]:
#| export
def make_connection(dir:str|None=None)->tuple[str,dict]:
    "Write a connection file with random free ports and a fresh key; returns `(path, info)`."
    dir = dir or mkdtemp(prefix='jupygate-')
    fname = str(Path(dir)/f"kernel-{uuid.uuid4().hex[:8]}.json")
    return write_connection_file(fname, ip='127.0.0.1', key=uuid.uuid4().hex.encode())

In [ ]:
cf, info = make_connection()
{k:v for k,v in info.items() if k!='key'}

{'shell_port': 56634,
 'iopub_port': 56635,
 'stdin_port': 56636,
 'control_port': 56637,
 'hb_port': 56638,
 'ip': '127.0.0.1',
 'transport': 'tcp',
 'signature_scheme': 'hmac-sha256',
 'kernel_name': ''}

### KernelProc


`KernelProc` starts one process from an argument template. It replaces `{connection_file}` with the generated filename, as kernelspecs do. These examples use `python -m ipymini`. Other compatible Jupyter kernel commands can use the same interface.

The child environment includes `JPY_PARENT_PID` with the gateway's pid. Kernels that monitor this variable, including ipymini, exit when the gateway dies. This protects against orphaned processes after a gateway crash. It depends on the kernel implementing that check.

A `username` adds a privilege wrapper to the command. By default this is `sudo -n -E -u`. `JUPYGATE_SUDO` can name a gosu-style helper instead. Solveit uses this to run kernels as an unprivileged user in Docker.

In [ ]:
#| export
def _sudo(username): 
    helper = os.environ.get('JUPYGATE_SUDO')
    return [helper, username] if helper else ['/usr/bin/sudo','-n','-E','-u',username]

class KernelProc:
    "One kernel process: spawn from an argv template, watch, terminate."
    def __init__(self, argv:list[str], env:dict|None=None, appendenv:dict|None=None, cwd:str|None=None, username:str|None=None):
        self.cfile, self.info = make_connection()
        cmd = [a.format(connection_file=self.cfile) for a in argv]
        if username:
            os.chmod(self.cfile, 0o644)
            cmd = [*_sudo(username), *cmd]
        full_env = dict(os.environ if env is None else env) | (appendenv or {})  # `env` replaces the inherited environment; `appendenv` overlays the base
        full_env['JPY_PARENT_PID'] = str(os.getpid())
        self.proc = subprocess.Popen(cmd, env=full_env, cwd=cwd, start_new_session=True)

    @property
    def pid(self): return self.proc.pid
    def alive(self)->bool: return self.proc.poll() is None
    def interrupt(self): os.kill(self.pid, signal.SIGINT)

    def terminate(self, timeout:float=5.0):
        "SIGTERM then SIGKILL; reaps the process and removes the connection file."
        if self.alive():
            self.proc.terminate()
            try: self.proc.wait(timeout)
            except subprocess.TimeoutExpired:
                self.proc.kill()
                self.proc.wait(timeout)
        Path(self.cfile).unlink(missing_ok=True)

In [ ]:
#| export
IPYMINI_ARGV = [sys.executable, '-m', 'ipymini', '-f', '{connection_file}']

In [ ]:
k = KernelProc(IPYMINI_ARGV)
k.alive(), k.pid != os.getpid()

(True, True)

A running process isn't necessarily ready to answer requests. Port binding and protocol readiness take more time. The next section checks those.

Here we test process cleanup. `terminate` stops the process, waits for it to exit, and removes its connection file:

In [ ]:
k.terminate()
test_eq(k.alive(), False)
test_eq(Path(k.cfile).exists(), False)
k.proc.returncode is not None

True

`alive()` also detects a process that exits by itself. It calls `poll`, which collects the exit status. `terminate` can still clean up that process's connection file:

In [ ]:
bad = KernelProc([sys.executable, '-c', 'raise SystemExit(3)'])
bad.proc.wait(5)
test_eq(bad.alive(), False)
bad.terminate()  # idempotent on a dead process
bad.proc.returncode

3

## Channels

Jupygate opens one zmq channel set per kernel process and keeps it open until that process ends. It uses DEALER sockets for `shell`, `control`, and `stdin`, plus a SUB socket for `iopub`. Connecting or disconnecting a websocket client doesn't replace these sockets.

Jupyter_server instead creates zmq streams for each websocket connection. It needs additional readiness probes and buffering handoffs when clients reconnect. Keeping the channels open avoids those repeated setup steps.

Jupygate uses `zmq.asyncio` and awaits sends. This avoids the synchronous shadow-socket sends used in jupyter_client's channel machinery. Those sends can consume the file-descriptor notification that an asynchronous receive needs. Conkernelclient documents and works around that failure.

`KernelChannels` opens the four message channels from the connection information. `beat` uses the fifth endpoint for heartbeat requests. `KernelProc.alive()` checks the operating-system process. A heartbeat checks whether the kernel's zmq endpoint responds.

The channel's `Session` uses the kernel key to sign outgoing messages and verify incoming messages. Websocket clients don't need that key.

The iopub SUB socket sets `RCVHWM=0`, disabling its default receive high-water mark of 1000 messages. A bounded zmq pipe can silently discard output, including `status` messages, when it fills. Losing `idle` can leave clients waiting indefinitely. The kernel's application-level policy of keeping statuses cannot prevent drops in a later socket queue.

Jupygate applies its output-drop policy in `ClientQueue` instead. Disabling the zmq bound prevents that additional source of message loss. It is not a total memory bound. The kernel's own output queue policy can limit its backlog, but the gateway's zmq receive queue remains unbounded.

All DEALER sockets use the same zmq identity. When code calls `input()`, the kernel sends its stdin request to the identity from the original shell request. Shell and stdin must therefore identify the gateway as the same peer. Jupyter_client uses the same convention.

In [ ]:
#| export
CHANNELS = ('shell','control','stdin','iopub')

class KernelChannels:
    "One persistent set of zmq.asyncio sockets connected to a kernel."
    def __init__(self, info:dict):
        self.session = Session(key=info['key'].encode() if isinstance(info['key'],str) else info['key'])
        self.ctx = zmq.asyncio.Context()
        addr = lambda port: f"{info['transport']}://{info['ip']}:{port}"
        self.socks = {}
        for name in CHANNELS:
            kind = zmq.SUB if name=='iopub' else zmq.DEALER
            s = self.ctx.socket(kind)
            s.linger = 0
            if kind==zmq.DEALER: s.setsockopt(zmq.IDENTITY, self.session.bsession)
            else:
                s.subscribe(b'')
                s.rcvhwm = 0  # before connect. Lossless hop: libzmq's HWM drops silently (statuses included); shedding belongs to ClientQueue alone
            s.connect(addr(info[f'{name}_port']))
            self.socks[name] = s
        self.hb_addr, self.hb_sock = addr(info['hb_port']), None

    async def send(self, channel:str, msg:dict):
        "Sign, serialize, and send a message dict on `channel`, awaiting the socket send so errors and backpressure surface here."
        m = dict(header=msg['header'], parent_header=msg.get('parent_header') or {}, metadata=msg.get('metadata') or {}, content=msg.get('content') or {})
        await self.socks[channel].send_multipart(self.session.serialize(m) + [memoryview(b) for b in msg.get('buffers') or []])

    async def recv(self, channel:str)->dict:
        "Receive and verify one message from `channel` (iopub topic frames are handled)."
        parts = await self.socks[channel].recv_multipart()
        idents, msg_list = self.session.feed_identities(parts)
        return self.session.deserialize(msg_list)

    async def beat(self, timeout:float=1.0)->bool:
        "One heartbeat ping; True if the kernel echoed in time. A missed reply resets the REQ socket (strict alternation)."
        if self.hb_sock is None:
            self.hb_sock = self.ctx.socket(zmq.REQ)
            self.hb_sock.linger = 0
            self.hb_sock.connect(self.hb_addr)
        await self.hb_sock.send(b'ping')
        if await self.hb_sock.poll(timeout*1000):
            await self.hb_sock.recv()
            return True
        self.hb_sock.close()
        self.hb_sock = None
        return False

    def close(self):
        for s in self.socks.values(): s.close()
        if self.hb_sock: self.hb_sock.close()
        self.ctx.term()

`send` serializes the message with `Session.serialize`, then awaits the asyncio socket's `send_multipart`. It does not call `Session.send` or send through a synchronous shadow socket. `recv` also uses the asyncio socket directly. The regression test in conkernelclient's repository covers the shadow-socket failure described above.

Let's connect to a real kernel. We'll reuse this process and its channels through the following sections.

In [ ]:
k = KernelProc(IPYMINI_ARGV)
ch = KernelChannels(k.info)
k.alive()

True

### The ready-wait


A new zmq SUB connection can miss messages before its subscription reaches the publisher. This is the slow-joiner problem. Ipymini and modern ipykernel implement JEP 65: the iopub XPUB socket sends `iopub_welcome` after registering a subscription. The welcome confirms that subscription is active.

The welcome can arrive before the shell router is ready. It doesn't prove the kernel will answer requests. The gateway must observe the welcome rather than negotiate support before subscribing.

`wait_ready` sends `kernel_info_request` probes while waiting for iopub traffic. Readiness requires a shell reply and evidence that iopub works.

When the first iopub message is `iopub_welcome`, the gateway sends one final probe as an end marker. It waits for that probe's reply and matching `idle` status. Earlier probes precede this marker on their channels. Consuming the marker therefore drains their traffic before handing the channels to the mux.

Any other first message selects the compatibility path for kernels without a welcome. It reads a `kernel_info_reply`, drains outstanding probe replies, then drains iopub until 0.2 seconds of silence. This path uses a quiet interval rather than an explicit end marker.

In [ ]:
#| export
async def wait_ready(ch:KernelChannels, timeout:float=30.0, probe_every:float=0.5)->dict:
    "Wait for a kernel_info reply and iopub traffic, drain readiness probes, and return the reply content."
    loop = asyncio.get_running_loop()
    end = loop.time() + timeout
    def left(cap=1.0):
        t = end - loop.time()
        if t <= 0: raise TimeoutError(f"kernel not ready after {timeout}s")
        return min(t, cap)
    async def recv_or_none(channel, cap=1.0):
        if await ch.socks[channel].poll(int(left(cap)*1000)): return await ch.recv(channel)
        return None
    probes = set()
    async def probe():
        m = ch.session.msg('kernel_info_request')
        probes.add(m['header']['msg_id'])
        await ch.send('shell', m)
        return m['header']['msg_id']
    msg = None
    while msg is None:
        await probe()
        msg = await recv_or_none('iopub', probe_every)
    if msg['header']['msg_type'] == 'iopub_welcome':
        mid = await probe()  # end marker: replies are FIFO, so once its reply and idle arrive, every earlier probe's traffic is consumed
        while (reply := await recv_or_none('shell')) is None or reply['parent_header'].get('msg_id') != mid: pass
        while not (msg['header']['msg_type']=='status' and msg['parent_header'].get('msg_id')==mid and msg['content']['execution_state']=='idle'):
            while (msg := await recv_or_none('iopub')) is None: pass
    else:
        while (reply := await recv_or_none('shell')) is None or reply['header']['msg_type'] != 'kernel_info_reply': pass
        outstanding = len(probes) - 1  # every probe gets a reply; one was consumed above
        while outstanding and await ch.socks['shell'].poll(1000):
            m = await ch.recv('shell')
            if m['parent_header'].get('msg_id') in probes: outstanding -= 1
        while await ch.socks['iopub'].poll(200): await ch.recv('iopub')  # then drain iopub until 0.2s of silence
    return reply['content']


In [ ]:
info = await wait_ready(ch)
assert not await ch.socks['iopub'].poll(300)  # ready leaves the channels clean: no probe traffic dribbles in afterwards
info['implementation'], info['status']


('ipymini', 'ok')

Setting `IPYMINI_IOPUB_XPUB=0` disables ipymini's welcome behavior. This exercises the compatibility path with probe traffic and a final quiet interval. The test checks that no iopub messages remain after readiness:

In [ ]:
k3 = KernelProc(IPYMINI_ARGV, env=dict(os.environ, IPYMINI_IOPUB_XPUB='0'))
ch3 = KernelChannels(k3.info)
info = await wait_ready(ch3)
assert not await ch3.socks['iopub'].poll(300)
ch3.close()
k3.terminate()
info['status']

'ok'

With welcome support enabled, a fresh ipymini subscription starts with `iopub_welcome`. This checks the subscription handshake on its own, without sending a shell request:

In [ ]:
k2 = KernelProc(IPYMINI_ARGV)
ch2 = KernelChannels(k2.info)
first = await asyncio.wait_for(ch2.recv('iopub'), 30)
test_eq(first['header']['msg_type'], 'iopub_welcome')
first['content']

{'subscription': ''}

In [ ]:
#| hide
ch2.close()
k2.terminate()

### A cell, end to end


Send an `execute_request` on `shell` to run a cell. Iopub reports `busy`, `execute_input`, printed output, then `idle`. The `execute_reply` arrives separately on `shell`. These are the messages the mux will distribute to websocket clients.

In [ ]:
req = ch.session.msg('execute_request', dict(code='print(6*7)', silent=False, store_history=True,
    user_expressions={}, allow_stdin=False, stop_on_error=True))
await ch.send('shell', req)
seq = []
while True:
    m = await asyncio.wait_for(ch.recv('iopub'), 10)
    if m['parent_header'].get('msg_id') != req['header']['msg_id']: continue
    seq.append(m['header']['msg_type'])
    if m['header']['msg_type']=='status' and m['content']['execution_state']=='idle': break
seq

['status', 'execute_input', 'stream', 'status']

In [ ]:
reply = await asyncio.wait_for(ch.recv('shell'), 10)
test_eq(reply['header']['msg_type'], 'execute_reply')
test_eq(reply['content']['status'], 'ok')
reply['content']['execution_count']

1

Ipymini and ipykernel accept an `interrupt_request` on the control channel. The HTTP API also offers a SIGINT interrupt for kernels configured to use signals:

In [ ]:
imsg = ch.session.msg('interrupt_request', {})
await ch.send('control', imsg)
ireply = await asyncio.wait_for(ch.recv('control'), 10)
test_eq(ireply['header']['msg_type'], 'interrupt_reply')
ireply['content']

{'status': 'ok'}

The heartbeat endpoint runs independently of cell execution. A busy kernel can still answer its heartbeat. A successful echo shows that the zmq endpoint responds, not that the kernel is idle or able to finish its current cell. `GatewayKernel` uses this check alongside process liveness.

In [ ]:
await ch.beat()

True

## The mux

`KernelMux` routes one kernel's traffic to multiple clients:

1. It broadcasts iopub messages to every client queue. Outputs are shared, as in Jupyter.
2. It routes shell and control replies using `parent_header.session`. This identifies the client session that sent the request. No request-id table is needed.
3. It routes stdin prompts to the requesting session. It can also supply missing parent headers in stdin replies.
4. Each client has its own outbound queue. A stalled client doesn't stop delivery to other clients.

An attached client's queue can drop non-`status` iopub output when full. It logs the drops and keeps statuses and replies. The policy resembles ipymini's iopub queue policy, but acts per client. It makes drops explicit instead of relying on zmq's silent high-water-mark behavior.

A client supplies a `session_id` and an asynchronous `send(frame)` callable. The HTTP app adapts websockets to this interface. The examples use lists to collect frames for inspection.

### Client queues


`ClientQueue` stores encoded frames in a deque. Its writer task awaits the client's `send`. If a websocket stalls, only that writer waits. New messages still enter its queue, subject to the output-drop policy.

The writer removes a frame only after `send` succeeds. If sending raises an exception, the frame stays queued for reconnection. Encoding happens before queueing. The iopub pump can encode once and share that frame across all client queues.

After every send, the writer yields to the event loop. Some buffered transports complete sends without suspending. An explicit yield gives cancellation and disconnect handlers time to run before the writer drains more queued output.

A successful send is not a delivery acknowledgment. The network can fail after the socket accepts a frame. This protocol has no acknowledgments or sequence numbers to recover that loss.


In [ ]:
#| export
class ClientQueue:
    "Outbound frames for one client; attached queues exempt statuses and non-iopub messages from the bound."
    def __init__(self, session_id:str, send, qmax:int=1000, buffer:bool=True):
        self.session_id, self.qmax, self.buffer = session_id, qmax, buffer
        self.q, self.dropped, self._drop_mark, self._wake = deque(), 0, 0, asyncio.Event()
        self.task = self.detached_at = None
        self.attach(send)

    @property
    def detached(self): return self.detached_at is not None

    def attach(self, send):
        "(Re)bind the queue to a live websocket: install `send` and start a fresh writer."
        if self.task: self.task.cancel()
        self._send, self.detached_at = send, None
        self.task = asyncio.create_task(self._writer())

    def detach(self):
        "Park the queue as a ring: stop the writer, remember when and how much was already dropped."
        if self.task: self.task.cancel()
        self.task, self.detached_at, self._drop_mark = None, time.monotonic(), self.dropped

    def put(self, msg:dict): self.put_frame(msg['header']['msg_type'], to_frame(msg), iopub=msg.get('channel')=='iopub')

    def put_frame(self, msg_type:str, frame:str|bytes, iopub:bool=True):
        if len(self.q) >= self.qmax and (self.detached or (iopub and msg_type!='status')):
            self.dropped += 1
            if self.dropped==1 or self.dropped%1000==0: log.warning("client %s queue full; dropped=%d", self.session_id, self.dropped)
            if not self.detached: return  # attached: the newcomer is dropped
            self.q.popleft()              # detached ring: the oldest is evicted, the newcomer kept
        self.q.append((msg_type, frame))
        self._wake.set()

    def force(self, msg:dict):
        "Append a frame without checking the bound; reattachment uses this for its warning and status."
        self.q.append((msg['header']['msg_type'], to_frame(msg)))
        self._wake.set()

    async def _writer(self):
        while True:
            if not self.q:
                self._wake.clear()
                await self._wake.wait()
            try: await self._send(self.q[0][1])
            except Exception: return  # connection gone; the frame stays queued for a reattach
            self.q.popleft()
            await asyncio.sleep(0)  # a buffered transport can complete sends without suspending; yield so cancel and disconnect can land

    def close(self):
        if self.task: self.task.cancel()

`KernelMux` runs a receive task for each channel. When buffering is enabled, it also runs a task that removes expired detached queues.

The iopub task encodes each message once and puts it in every client queue. It records the latest `execution_state` for reconnect status reports. Other channel tasks look up the client from `parent_header.session`. Messages for an unknown session have no recipient and are discarded. Examples include replies to gateway readiness probes and replies for clients whose queues no longer exist.


In [ ]:
#| export
class KernelMux:
    "Fan kernel traffic out to clients; route client frames back to kernel channels."
    def __init__(self, ch:KernelChannels, qmax:int=1000, buffer_secs:float=3600.0):
        self.ch, self.qmax, self.buffer_secs = ch, qmax, buffer_secs
        self.clients, self.pending_stdin, self.exec_state = {}, {}, 'starting'
        self.tasks = [asyncio.create_task(self._pump(c)) for c in CHANNELS]
        if buffer_secs: self.tasks.append(asyncio.create_task(self._reaper()))

    @property
    def n_attached(self): return sum(1 for cq in self.clients.values() if not cq.detached)

    def add(self, session_id:str, send, buffer:bool=True)->ClientQueue:
        "Register a client, or reattach a returning session to its surviving queue."
        cq = self.clients.get(session_id)
        if cq is None:
            self.clients[session_id] = cq = ClientQueue(session_id, send, self.qmax, buffer=buffer)
            return cq
        missed = cq.dropped - cq._drop_mark if cq.detached else 0
        cq.attach(send)
        if missed:
            warn = self.ch.session.msg('stream', dict(name='stderr', text=f'[jupygate] {missed} messages dropped while disconnected\n'))
            for m in (warn, self.ch.session.msg('status', dict(execution_state=self.exec_state))):
                m['channel'] = 'iopub'
                cq.force(m)
        return cq

    def drop(self, session_id:str):
        "A websocket went away: park the queue if the session is bufferable, else discard it."
        cq = self.clients.get(session_id)
        if cq is None: return
        if self.buffer_secs and cq.buffer: cq.detach()
        else:
            self.clients.pop(session_id)
            cq.close()

    async def _reaper(self):
        while True:
            await asyncio.sleep(min(self.buffer_secs, 60))
            cutoff = time.monotonic() - self.buffer_secs
            for sid, cq in list(self.clients.items()):
                if cq.detached and cq.detached_at < cutoff:
                    self.clients.pop(sid)
                    cq.close()

    async def _pump(self, channel:str):
        while True:
            msg = await self.ch.recv(channel)
            await asyncio.sleep(0)  # zmq recv on a buffered socket completes without suspending; yield so writers and handlers run mid-flood
            msg['channel'] = channel
            if channel=='iopub':
                if msg['header']['msg_type']=='status': self.exec_state = msg['content']['execution_state']
                mt, frame = msg['header']['msg_type'], to_frame(msg)
                for cq in self.clients.values(): cq.put_frame(mt, frame)
                continue
            sid = msg['parent_header'].get('session')
            if channel=='stdin' and msg['header']['msg_type']=='input_request': self.pending_stdin[sid] = msg['header']
            if (cq := self.clients.get(sid)): cq.put(msg)

    async def handle_frame(self, session_id:str, data:str|bytes):
        "One frame from a client websocket: decode, repair stdin parents, send to the kernel."
        msg = from_frame(data)
        channel = msg.pop('channel', 'shell')
        if channel=='stdin' and not msg.get('parent_header') and (hdr := self.pending_stdin.pop(session_id, None)):
            msg['parent_header'] = hdr
        await self.ch.send(channel, msg)

    def synthesize_status(self, state:str):
        "Broadcast a gateway-made status (`restarting`/`dead`): kernels cannot announce their own death."
        self.exec_state = state
        msg = self.ch.session.msg('status', dict(execution_state=state))
        msg['channel'] = 'iopub'
        mt, frame = 'status', to_frame(msg)
        for cq in self.clients.values(): cq.put_frame(mt, frame)

    def close(self):
        for t in self.tasks: t.cancel()
        for cq in self.clients.values(): cq.close()
        self.clients.clear()

### Two clients, one kernel


We'll attach two simulated websocket clients to the running kernel. Each has a `Session` for generating headers and a list for received frames. The helpers build execute requests and wait for expected output:


In [ ]:
def mk_client():
    "A simulated websocket client with a Session for headers and a list of decoded frames."
    ses, got = Session(key=b'unused'), []
    async def send(frame): got.append(from_frame(frame))
    return ses, got, send

def exec_frame(ses, code):
    msg = ses.msg('execute_request', dict(code=code, silent=False, store_history=True,
        user_expressions={}, allow_stdin=True, stop_on_error=False))
    msg['channel'] = 'shell'
    return msg

async def until(pred, timeout=15):
    "Poll the fake clients until `pred()` is true."
    async with asyncio.timeout(timeout):
        while not pred(): await asyncio.sleep(0.02)


Websocket clients send message dictionaries without HMAC signatures. The keys passed to these clients' `Session` objects are unused. The gateway signs messages with the kernel key when forwarding them over zmq. It never sends that key to websocket clients.

Both clients submit code. Each sees its own reply and both cells' iopub output:

In [ ]:
mux = KernelMux(ch)
ses_a, got_a, send_a = mk_client()
ses_b, got_b, send_b = mk_client()
mux.add(ses_a.session, send_a)
mux.add(ses_b.session, send_b)
await mux.handle_frame(ses_a.session, to_frame(exec_frame(ses_a, "a_val = 'from A'; print(a_val)")))
await mux.handle_frame(ses_b.session, to_frame(exec_frame(ses_b, "print('from B')")))
await until(lambda: any(m['channel']=='shell' for m in got_a) and any(m['channel']=='shell' for m in got_b))
[m['header']['msg_type'] for m in got_a if m['channel']=='shell'], [m['header']['msg_type'] for m in got_b if m['channel']=='shell']


(['execute_reply'], ['execute_reply'])

In [ ]:
replies_a = [m for m in got_a if m['channel']=='shell']
test_eq(replies_a[0]['parent_header']['session'], ses_a.session)  # A got A's reply, not B's
streams = {m['content']['text'].strip() for m in got_a if m['header']['msg_type']=='stream'}
assert {'from A','from B'} <= streams  # iopub is broadcast: A saw B's print too
streams

{'from A', 'from B'}

### Stdin and the parent-stamping repair


A call to `input()` sends an `input_request` to the client running that cell. Some clients, including jupyter_client, omit the parent header in their `input_reply`. Kernels then rely on zmq identities to associate the reply with a prompt. Shared gateway channels cannot distinguish clients by those identities.

The mux remembers the last `input_request` header for each client session. If an incoming stdin reply has no parent, `handle_frame` inserts that saved header. Replies with an existing parent remain unchanged. Our websocket client supplies the parent itself, but the repair supports clients that don't.

In [ ]:
await mux.handle_frame(ses_a.session, to_frame(exec_frame(ses_a, "answer = input('name? ')")))
await until(lambda: any(m['header']['msg_type']=='input_request' for m in got_a))
prompt = next(m for m in got_a if m['header']['msg_type']=='input_request')
prompt['content']

{'prompt': 'name? ', 'password': False}

In [ ]:
reply = ses_a.msg('input_reply', dict(value='Jeremy'))
reply['channel'] = 'stdin'   # note: no parent_header set - the worst-case client
await mux.handle_frame(ses_a.session, to_frame(reply))
await until(lambda: len([m for m in got_a if m['channel']=='shell']) >= 2)
await mux.handle_frame(ses_a.session, to_frame(exec_frame(ses_a, "print(answer)")))
await until(lambda: any('Jeremy' in m['content'].get('text','') for m in got_a if m['header']['msg_type']=='stream'))
next(m['content']['text'] for m in got_a if m['header']['msg_type']=='stream' and 'Jeremy' in m['content']['text'])

'Jeremy\n'

### Overflow: what a slow client misses, and what it never misses


We'll stop one queue's writer and reduce its bound to five frames. This simulates a stalled websocket during an output flood.

While attached, the bound rejects excess non-`status` iopub messages. It always accepts statuses and shell, control, and stdin messages. Those channels carry replies and prompts driven by client requests. Dropping an `execute_reply` would leave its caller waiting for a reply that cannot arrive.

The slow client can miss printed output but retain the `busy` and `idle` statuses and its execution reply:


In [ ]:
ses_c, got_c, send_c = mk_client()
cq = mux.add(ses_c.session, send_c)
cq.task.cancel()      # stall the writer: frames pile up in the deque
cq.qmax = 5
await mux.handle_frame(ses_c.session, to_frame(exec_frame(ses_c, "for i in range(200): print(i)")))
await until(lambda: cq.dropped > 0 and sum(t=='status' for t,_ in cq.q) >= 2)
types = [t for t,_ in cq.q]
cq.dropped, types.count('status'), len(cq.q) <= 5 + sum(t!='stream' for t in types), 'execute_reply' in types


client 49766e3f-63c2-43a8-8299-36d509172ca5 queue full; dropped=1


(197, 2, True, True)

### Reconnects

When a buffered client disconnects, `drop` stops its queue's writer but keeps the deque. Kernel traffic continues to enter this detached queue. Reconnecting with the same `session_id` attaches a new send function to it. Buffered messages precede new traffic because they share one queue.

The websocket API enables buffering only when the client supplies `session_id`. A client cannot reconnect using an id it never knew. Jupyasyncclient supplies its own id. In-process clients can choose whether to buffer with the mux's `buffer` argument.

While detached, the queue applies its bound to all messages, including statuses and replies. It evicts old frames to retain recent traffic. An attached queue's exemptions would otherwise allow indefinite growth while nobody reads it. If the queue was already over its nominal bound when detached, replacement retains that length rather than shrinking it immediately.

Detached queues expire after `buffer_secs`, which defaults to one hour. Setting it to `0` disables buffering.

If frames were lost while detached, reattachment appends two messages after the remaining output. A stderr `stream` reports how many messages were dropped. A `status` reports the current execution state. These messages bypass the normal queue bound.

This warning cannot replace a lost reply. Clients need their own timeouts for requests whose replies were evicted. Buffering also cannot recover frames that a socket accepted before an undetected network failure.

Client `a` disconnects while client `b` runs another cell. No new frames reach `a`'s old send function. Its detached queue still stores the output:

In [ ]:
mux.drop(ses_a.session)
n = len(got_a)
await mux.handle_frame(ses_b.session, to_frame(exec_frame(ses_b, "'while a is away'")))
await until(lambda: any(m['header']['msg_type']=='execute_result' for m in got_b))
qa = mux.clients[ses_a.session]
test_eq(len(got_a), n)
qa.detached, len(qa.q) > 0

(True, True)

Calling `add` with the same session id installs `a`'s replacement send function. Its queue drains the buffered cell output, including `execute_result`:

In [ ]:
got_a2 = []
async def send_a2(frame): got_a2.append(from_frame(frame))
mux.add(ses_a.session, send_a2)
await until(lambda: any(m['header']['msg_type']=='execute_result' for m in got_a2))
next(m['content']['data']['text/plain'] for m in got_a2 if m['header']['msg_type']=='execute_result')

"'while a is away'"

Client `c` still has the small queue bound from the overflow example. During disconnection it loses old messages. On reattachment, the final two frames report the number lost and the current execution state:

In [ ]:
mux.drop(ses_c.session)
nb = len(got_b)
await mux.handle_frame(ses_c.session, to_frame(exec_frame(ses_c, "for i in range(200): print(i)")))
await until(lambda: any(m['header']['msg_type']=='status' and m['content']['execution_state']=='idle' for m in got_b[nb:]))
got_c2 = []
async def send_c2(frame): got_c2.append(from_frame(frame))
cq = mux.add(ses_c.session, send_c2)
await until(lambda: not cq.q)
warn = next(m for m in got_c2 if m['header']['msg_type']=='stream' and 'dropped while disconnected' in m['content']['text'])
test_eq(got_c2[-1]['header']['msg_type'], 'status')
len(got_c2), warn['content']['text'].strip()

(11, '[jupygate] 204 messages dropped while disconnected')

The gateway can broadcast a lifecycle status itself. A dead kernel cannot send its own `dead` status:

In [ ]:
mux.synthesize_status('dead')
await until(lambda: any(m['content'].get('execution_state')=='dead' for m in got_a2 if m['header']['msg_type']=='status'))
sum(1 for got in (got_a2, got_b) for m in got if m['content'].get('execution_state')=='dead')

2

In [ ]:
#| hide
mux.close()
ch.close()
k.terminate()

## The gateway server

The Starlette app exposes jupyter_server's kernel endpoints. Each kernel id maps to a `GatewayKernel`, which combines its process, channels, and mux.

The kernel routes are:

- `GET /api/kernels` lists kernels.
- `POST /api/kernels` creates a kernel.
- `GET /api/kernels/{id}` returns its model.
- `DELETE /api/kernels/{id}` shuts it down.
- `POST /api/kernels/{id}/interrupt` interrupts it.
- `POST /api/kernels/{id}/restart` restarts it.
- `WS /api/kernels/{id}/channels?session_id=...` connects a client to its mux.

The server returns JSON and websocket frames. It has no HTML interface, files API, or kernelspec lookup. Clients manage files by running code in a kernel. Creation accepts explicit arguments, environment, and working directory.

Pass `auth_token` to require authentication. Requests can supply `Authorization: token ...` or `?token=...`. Both forms work for HTTP and websocket requests. Without a token, the service is open. Use that mode only in a trusted local environment.

### GatewayKernel


`GatewayKernel` combines process management, channels, and routing. Each call to `start` runs the readiness check for its new process before starting the mux.

`_watch` polls process liveness and heartbeats. Three consecutive missed heartbeats change the model's state from `alive` to `unresponsive`. A later echo restores `alive`. The gateway never kills a kernel merely because its heartbeat stopped.

If the process exits unexpectedly, `_watch` marks it `dead` and broadcasts that status. Explicit shutdown also marks it `dead` while terminating the process. These lifecycle states are separate from the kernel's `busy` and `idle` execution messages.

`restart` terminates the old process and starts another with fresh ports and channels. Clients see `restarting`, then `starting` after the new process passes readiness.


In [ ]:
#| export
class GatewayKernel:
    "A kernel process plus its channel set and mux, as one lifecycle."
    def __init__(self, argv:list[str], env:dict|None=None, appendenv:dict|None=None, cwd:str|None=None,
        username:str|None=None, qmax:int=1000, buffer_secs:float=3600.0):
        self.spec = dict(argv=argv, env=env, appendenv=appendenv, cwd=cwd, username=username)
        self.qmax, self.buffer_secs = qmax, buffer_secs
        self.id = uuid.uuid4().hex
        self.proc = self.ch = self.mux = self._watcher = None
        self.state = 'starting'
        self.last_beat = None

    async def start(self, timeout:float=60.0):
        self.proc = KernelProc(**self.spec)
        self.ch = KernelChannels(self.proc.info)
        await wait_ready(self.ch, timeout)
        self.mux = KernelMux(self.ch, qmax=self.qmax, buffer_secs=self.buffer_secs)
        self.state = 'alive'
        self._watcher = asyncio.create_task(self._watch())
        return self

    async def _watch(self):
        misses = 0
        while self.proc.alive():
            if await self.ch.beat():
                self.last_beat, misses = time.time(), 0
                if self.state=='unresponsive':
                    log.warning(f'kernel {self.id}: responsive again')
                    self.state = 'alive'
            else:
                misses += 1
                if misses>=3 and self.state=='alive':
                    log.warning(f'kernel {self.id}: unresponsive, 3 heartbeats missed')
                    self.state = 'unresponsive'
            await asyncio.sleep(1.0)
        if self.state in ('alive','unresponsive'):
            self.state = 'dead'
            self.mux.synthesize_status('dead')

    def model(self)->dict: return dict(id=self.id, name='', execution_state=self.state, connections=self.mux.n_attached if self.mux else 0,
        pid=self.proc.pid if self.proc else None, last_heartbeat=self.last_beat)
    def interrupt(self): self.proc.interrupt()

    async def restart(self, timeout:float=60.0):
        self.state = 'restarting'
        self.mux.synthesize_status('restarting')
        old_mux = self.mux
        clients, old_mux.clients = old_mux.clients, {}
        await self.shutdown(keep_state=True)
        old_mux.close()
        await self.start(timeout)
        self.mux.clients = clients
        self.mux.synthesize_status('starting')

    async def shutdown(self, keep_state:bool=False):
        if self._watcher: self._watcher.cancel()
        if not keep_state:
            self.state = 'dead'
            if self.mux: self.mux.close()
        if self.ch: self.ch.close()
        if self.proc: await asyncio.to_thread(self.proc.terminate)

Restart preserves the client queues and their send functions. The new mux takes over those queues without requiring websocket clients to reconnect.

The gateway sends `restarting` before shutdown. It consumes the new kernel's readiness traffic, including its welcome, before attaching the client queues. Then it broadcasts `starting`. Clients see gateway-generated lifecycle messages instead of the kernel's boot probes.


### The kernel registry

`Kernels` manages `GatewayKernel` instances independently of HTTP. An embedding application, such as an MCP server, can create a kernel and attach directly to its mux. It uses the same reply routing, stdin repair, and lifecycle statuses as a websocket client.

The HTTP app adds routes and authentication around this registry. If `create` fails during startup, it removes the entry and shuts down its resources before raising the error. Failed startup does not leave a partial kernel in the registry.

In [ ]:
#| export
class Kernels:
    "Kernel registry: create, look up, and reap `GatewayKernel`s. The HTTP app and embedders both drive this."
    def __init__(self, argv:list[str]=IPYMINI_ARGV, qmax:int=1000, buffer_secs:float=3600.0):
        self.argv, self.qmax, self.buffer_secs = argv, qmax, buffer_secs
        self.kernels = {}

    async def create(self, argv:list[str]|None=None, env:dict|None=None, appendenv:dict|None=None,
        cwd:str|None=None, username:str|None=None, timeout:float=60.0)->GatewayKernel:
        gk = GatewayKernel(argv=argv or self.argv, env=env, appendenv=appendenv, cwd=cwd, username=username, qmax=self.qmax, buffer_secs=self.buffer_secs)
        self.kernels[gk.id] = gk
        try: await gk.start(timeout)
        except Exception:
            self.kernels.pop(gk.id, None)
            await gk.shutdown()
            raise
        return gk

    async def delete(self, kid:str):
        gk = self.kernels.pop(kid)
        await gk.shutdown()

    async def shutdown(self):
        "Attempt shutdown of every registered kernel, then clear the registry."
        await asyncio.gather(*[k.shutdown() for k in self.kernels.values()], return_exceptions=True)
        self.kernels.clear()

    def get(self, kid:str)->GatewayKernel|None: return self.kernels.get(kid)
    def values(self): return self.kernels.values()
    def __getitem__(self, kid:str)->GatewayKernel: return self.kernels[kid]
    def __len__(self): return len(self.kernels)

In [ ]:
kernels = Kernels()
gk = await kernels.create()
ses, got, send = mk_client()
gk.mux.add(ses.session, send, buffer=False)   # an in-process client: no websocket, no reconnect story
await gk.mux.handle_frame(ses.session, to_frame(exec_frame(ses, "6*7")))
await until(lambda: any(m['channel']=='shell' for m in got))
test_eq(kernels[gk.id], gk)
await kernels.shutdown()
test_eq(len(kernels), 0)

In [ ]:
#| export
def _authed(request, token):
    if not token: return True
    hdr = request.headers.get('authorization', '')
    return hdr == f'token {token}' or request.query_params.get('token') == token

def create_app(argv:list[str]=IPYMINI_ARGV, auth_token:str|None=None, qmax:int=1000, buffer_secs:float=3600.0,
    term_cull_timeout:float=0)->Starlette:
    "The gateway app: kernels plus terminals. `argv` is the default kernel command; creation requests may override it."
    kernels = Kernels(argv, qmax=qmax, buffer_secs=buffer_secs)
    from ptymini.core import PtyRegistry, cull_loop
    from jupygate.term import term_routes
    terminals = PtyRegistry(cull_timeout=term_cull_timeout)

    def _kernel(request):
        k = kernels.get(request.path_params['kid'])
        if k is None: raise KeyError
        return k

    async def list_kernels(request): return JSONResponse([k.model() for k in kernels.values()])

    async def create_kernel(request):
        body = await request.json() if await request.body() else {}
        try: gk = await kernels.create(argv=body.get('argv'), env=body.get('env'), appendenv=body.get('appendenv'), cwd=body.get('cwd'), username=body.get('username'))
        except Exception as e: return JSONResponse(dict(message=f'kernel failed to start: {e}'), status_code=500)
        return JSONResponse(gk.model(), status_code=201)

    async def get_kernel(request): return JSONResponse(_kernel(request).model())

    async def delete_kernel(request):
        await kernels.delete(request.path_params['kid'])
        return Response(status_code=204)

    async def interrupt(request):
        _kernel(request).interrupt()
        return Response(status_code=204)

    async def restart(request):
        gk = _kernel(request)
        await gk.restart()
        return JSONResponse(gk.model())

    async def channels(ws):
        if not _authed(ws, auth_token): return await ws.send_denial_response(JSONResponse(dict(message='forbidden'), status_code=403))
        gk = kernels.get(ws.path_params['kid'])
        if gk is None or gk.mux is None: return await ws.send_denial_response(JSONResponse(dict(message='no such kernel'), status_code=404))
        sid = ws.query_params.get('session_id')
        sid, buffer = sid or uuid.uuid4().hex, sid is not None
        await ws.accept()
        async def send(frame):
            if isinstance(frame, bytes): await ws.send_bytes(frame)
            else: await ws.send_text(frame)
        gk.mux.add(sid, send, buffer=buffer)
        try:
            while True:
                event = await ws.receive()
                if event['type']=='websocket.disconnect': break
                try: await gk.mux.handle_frame(sid, event.get('bytes') or event['text'])
                except Exception as e: log.warning('dropped frame from %s: %s', sid, e)
        except WebSocketDisconnect: pass
        finally: gk.mux.drop(sid)

    def guard(fn):
        async def inner(request):
            if not _authed(request, auth_token): return JSONResponse(dict(message='forbidden'), status_code=403)
            try: return await fn(request)
            except KeyError: return JSONResponse(dict(message='no such kernel'), status_code=404)
        return inner

    @asynccontextmanager
    async def lifespan(app):
        cull = asyncio.create_task(cull_loop(terminals)) if term_cull_timeout else None
        yield
        if cull: cull.cancel()
        await kernels.shutdown()
        await terminals.shutdown()

    r = lambda p,meth,f: Route('/api/kernels'+p, guard(f), methods=[meth])
    app = Starlette(lifespan=lifespan, routes=[r('','GET',list_kernels), r('','POST',create_kernel), r('/{kid}','GET',get_kernel),
        r('/{kid}','DELETE',delete_kernel), r('/{kid}/interrupt','POST',interrupt), r('/{kid}/restart','POST',restart),
        WebSocketRoute('/api/kernels/{kid}/channels', channels), *term_routes(terminals, auth_token)])
    app.state.kernels = kernels
    app.state.terminals = terminals
    return app

### A live gateway


`serve(..., in_thread=True)` runs uvicorn in a daemon thread with its own event loop, as solveit's notebooks do for FastHTML apps. The following test gateway uses a free local port and runs until the final cleanup cell.

In [ ]:
#| export
def serve(app, host:str='127.0.0.1', port:int=8787, log_level:str='warning', in_thread:bool=False):
    "Run the gateway under uvicorn; `in_thread=True` waits until it is listening and returns the server with `server.url` set (`port=0` picks a free port)."
    import threading, time, uvicorn
    server = uvicorn.Server(uvicorn.Config(app, host=host, port=port, log_level=log_level, ws_max_size=64*2**20))
    if not in_thread:
        server.run()
        return server
    server.thread = threading.Thread(target=server.run, daemon=True)
    server.thread.start()
    end = time.monotonic() + 10
    while not server.started:
        if time.monotonic() > end: raise TimeoutError('uvicorn did not start')
        time.sleep(0.01)
    server.url = f'http://{host}:{server.servers[0].sockets[0].getsockname()[1]}'
    return server

def _env_app():
    "App factory for the reloader: its fresh workers re-import this module, so config rides in the environment; the token is popped so spawned kernels never inherit it"
    return create_app(auth_token=os.environ.pop('JUPYGATE_TOKEN', None))

def _reload_file():
    "The restart lever: touching this file restarts the gateway, killing all kernels; created at startup so it is always touchable"
    from fastcore.xdg import xdg_state_home
    p = xdg_state_home()/'jupygate'/'reload'/'r.py'
    p.parent.mkdir(parents=True, exist_ok=True)
    p.touch()
    return p

def main():
    "Console entry point: `jupygate [--port N] [--token T] [--log-level L] [--reload]`."
    import argparse, uvicorn
    from uvicorn.supervisors import ChangeReload
    p = argparse.ArgumentParser(description='Websocket gateway for Jupyter kernels')
    p.add_argument('--host', default='127.0.0.1')
    p.add_argument('--port', type=int, default=8787)
    p.add_argument('--token', default=None)
    p.add_argument('--log-level', default='warning', choices=['critical','error','warning','info','debug','trace'])
    p.add_argument('--reload', action='store_true', help='also restart on package source changes (dev)')
    args = p.parse_args()
    if args.token: os.environ['JUPYGATE_TOKEN'] = args.token
    reload_dirs = [str(_reload_file().parent)]
    if args.reload: reload_dirs.append(str(Path(__file__).parent))
    config = uvicorn.Config('jupygate.core:_env_app', factory=True, reload=True, reload_dirs=reload_dirs,
        host=args.host, port=args.port, log_level=args.log_level, ws_max_size=64*2**20, timeout_graceful_shutdown=5)
    server = uvicorn.Server(config)
    try: ChangeReload(config, target=server.run, sockets=[config.bind_socket()]).run()
    except KeyboardInterrupt: pass

In [ ]:
server = serve(create_app(), port=0, in_thread=True)
base = server.url
server.started, base


(True, 'http://127.0.0.1:56703')

Our test gateway starts with no kernels. Creating one launches a real ipymini process and waits for readiness. We then fetch its model:

In [ ]:
http = httpx.Client(base_url=base, timeout=90)
test_eq(http.get('/api/kernels').json(), [])
kid = http.post('/api/kernels').json()['id']
http.get(f'/api/kernels/{kid}').json()

{'id': 'aee4e064ba7c4e9b8720f23cd019d9bb',
 'name': '',
 'execution_state': 'alive',
 'connections': 0,
 'pid': 37689,
 'last_heartbeat': 1788943419.4315038}

The websocket uses the codecs from the first section. Each channel preserves message order. There is no ordering guarantee between channels, as with raw zmq. An execution reply can arrive before the final iopub output. This example waits for both the result and the reply:

In [ ]:
ses = Session(key=b'unused')
ws = ws_connect(f"{base.replace('http','ws')}/api/kernels/{kid}/channels?session_id={ses.session}")
msg = ses.msg('execute_request', dict(code='21*2', silent=False, store_history=True,
    user_expressions={}, allow_stdin=False, stop_on_error=True))
msg['channel'] = 'shell'
ws.send(to_frame(msg))
result = reply = None
while not (result and reply):
    m = from_frame(ws.recv(timeout=15))
    if m['header']['msg_type']=='execute_result': result = m['content']['data']['text/plain']
    if m['channel']=='shell': reply = m['content']['status']
result, reply


('42', 'ok')

Clients can interrupt through the websocket's control channel or the HTTP endpoint. The HTTP route sends SIGINT. This supports clients without a control channel and kernels that use signal interrupts.

Here, we use HTTP to stop an infinite loop in our disposable kernel:

In [ ]:
msg = ses.msg('execute_request', dict(code='import time\nwhile True: time.sleep(0.1)', silent=False,
    store_history=True, user_expressions={}, allow_stdin=False, stop_on_error=True))
msg['channel'] = 'shell'
ws.send(to_frame(msg))
time.sleep(0.5)
test_eq(http.post(f'/api/kernels/{kid}/interrupt').status_code, 204)
while True:
    m = from_frame(ws.recv(timeout=15))
    if m['channel']=='shell': break
m['content']['status'], m['content'].get('ename')

('error', 'KeyboardInterrupt')

The same websocket remains open during restart. We wait for `restarting` followed by `starting`, which means the replacement process passed readiness:


In [ ]:
test_eq(http.post(f'/api/kernels/{kid}/restart').status_code, 200)
states = []
while 'starting' not in states:
    m = from_frame(ws.recv(timeout=30))
    if m['header']['msg_type']=='status': states.append(m['content']['execution_state'])
assert 'restarting' in states
states[-2:]


['restarting', 'starting']

In [ ]:
ws.close()
test_eq(http.delete(f'/api/kernels/{kid}').status_code, 204)
test_eq(http.get('/api/kernels').json(), [])

`env` replaces the inherited kernel environment. This supports a controlled environment when running as another user. `appendenv` overlays the chosen base: the supplied `env`, or the gateway's environment if `env` is absent.

A remote client cannot read the gateway's environment to build a merged dictionary itself. With `appendenv`, it can request one additional variable and let the gateway preserve the rest:

In [ ]:
kid = http.post('/api/kernels', json=dict(appendenv=dict(JUPYGATE_DEMO_MARK='overlaid'))).json()['id']
ses2 = Session(key=b'unused')
ws = ws_connect(f"{base.replace('http','ws')}/api/kernels/{kid}/channels?session_id={ses2.session}")
msg = ses2.msg('execute_request', dict(code='import os; (os.environ["JUPYGATE_DEMO_MARK"], "PATH" in os.environ)',
    silent=False, store_history=True, user_expressions={}, allow_stdin=False, stop_on_error=True))
msg['channel'] = 'shell'
ws.send(to_frame(msg))
result = None
while not result:
    m = from_frame(ws.recv(timeout=15))
    if m['header']['msg_type']=='execute_result': result = m['content']['data']['text/plain']
ws.close()
test_eq(http.delete(f'/api/kernels/{kid}').status_code, 204)
test_eq(result, "('overlaid', True)")

### Auth


When `auth_token` is set, missing or incorrect credentials produce HTTP `403`. Both HTTP and websocket requests accept the token in a header or query parameter. Authentication happens before resource lookup.

Websocket requests for a missing kernel or terminal return HTTP `404`. These errors reject the opening handshake; the gateway doesn't accept a websocket session just to close it. This follows the [WebSocket protocol's handshake rules](https://www.rfc-editor.org/rfc/rfc6455.html#section-4.2.2). Below, the same missing-kernel URL returns `403` without credentials and `404` with the correct token.

In [ ]:
server2 = serve(create_app(auth_token='sekret'), port=0, in_thread=True)
open_http = httpx.Client(base_url=server2.url, timeout=30)
test_eq(open_http.get('/api/kernels').status_code, 403)
test_eq(open_http.get('/api/kernels', headers={'Authorization':'token sekret'}).status_code, 200)
test_eq(open_http.get('/api/kernels', headers={'Authorization':'token wrong'}).status_code, 403)
test_eq(open_http.get('/api/kernels', params={'token':'sekret'}).status_code, 200)

missing_ws = server2.url.replace('http', 'ws') + '/api/kernels/missing/channels'
with expect_fail(InvalidStatus, 'HTTP 403'): ws_connect(missing_ws)
with expect_fail(InvalidStatus, 'HTTP 404'): ws_connect(missing_ws, additional_headers={'Authorization':'token sekret'})
open_http.get('/api/kernels', headers={'Authorization':'token sekret'}).json()

ERROR:    ASGI callable returned without completing handshake.
ERROR:    ASGI callable returned without completing handshake.
ERROR:    ASGI callable returned without completing handshake.
ERROR:    ASGI callable returned without completing handshake.
ERROR:    ASGI callable returned without completing handshake.
ERROR:    ASGI callable returned without completing handshake.
ERROR:    ASGI callable returned without completing handshake.
ERROR:    ASGI callable returned without completing handshake.
ERROR:    ASGI callable returned without completing handshake.
ERROR:    ASGI callable returned without completing handshake.
ERROR:    ASGI callable returned without completing handshake.
ERROR:    ASGI callable returned without completing handshake.


In [ ]:
#| hide
server.should_exit = server2.should_exit = True

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()